# Lab 4 — Asistente de compras por voz

MVP local que recibe una consulta, busca productos y responde con texto y voz.

Flujo: datos ESCI → búsqueda RAG → agente → carrito y pedido → síntesis de voz.

El código controla las búsquedas, validaciones y operaciones de la tienda. El modelo redacta la respuesta usando el contexto recuperado.

> Ejecutar desde el entorno virtual del proyecto. La GPU es opcional; usa solo audios propios o autorizados.


## 1. Entorno y configuración

In [1]:
# Instala todas las dependencias al inicio del notebook.
!pip -q install pandas pyarrow sentence-transformers faiss-cpu scikit-learn transformers accelerate faster-whisper TTS

import os, random, json, time, re, uuid, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# Dispositivo compartido por ASR, TTS y carga de perfiles.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Busca la raíz del proyecto desde la carpeta actual o sus padres.
PROJECT_DIR = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'data' / 'raw').is_dir()),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError("Ejecuta el notebook dentro del proyecto: no se encontró data/raw.")

DATA_DIR = PROJECT_DIR / 'data' / 'raw'
ARTIFACTS = PROJECT_DIR / 'data' / 'artifacts'
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print('Carpeta del proyecto:', PROJECT_DIR)
print('Datos:', DATA_DIR)
print('Artefactos:', ARTIFACTS)


Dispositivo: cuda
Carpeta del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz
Datos: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/data/raw
Artefactos: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/data/artifacts


## 2. Cargar y explorar ESCI
Lee los Parquet existentes en `data/raw/`. `products` es el catálogo y `examples` contiene consultas, productos y juicios de relevancia.

In [2]:
products_path = DATA_DIR / 'shopping_queries_dataset_products.parquet'
examples_path = DATA_DIR / 'shopping_queries_dataset_examples.parquet'

if not (products_path.is_file() and examples_path.is_file()):
    raise FileNotFoundError(
        f'Faltan los Parquet en {DATA_DIR.resolve()}. '
        'Colócalos en data/raw antes de ejecutar esta celda.'
    )

products = pd.read_parquet(products_path)
examples = pd.read_parquet(examples_path)
print(products.shape, products.columns.tolist())
print(examples.shape, examples.columns.tolist())
print(products.product_locale.value_counts(dropna=False))
print(examples.esci_label.value_counts(dropna=False))
display(products.head(3)); display(examples.head(3))


(1814924, 7) ['product_id', 'product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color', 'product_locale']
(2621288, 9) ['example_id', 'query', 'query_id', 'product_id', 'product_locale', 'esci_label', 'small_version', 'large_version', 'split']
us    1215854
jp     339059
es     260011
Name: product_locale, dtype: int64
E    1708158
S     574313
I     263165
C      75652
Name: esci_label, dtype: int64


,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,B079VKKJN7,"11 Degrees de los Hombres Playera con Logo, Ne...",Esta playera con el logo de la marca Carrier d...,11 Degrees Negro Playera con logo\nA estrenar ...,11 Degrees,Negro,es
1,B079Y9VRKS,Camiseta Eleven Degrees Core TS White (M),None,None,11 Degrees,Blanco,es
2,B07DP4LM9H,11 Degrees de los Hombres Core Pull Over Hoodi...,La sudadera con capucha Core Pull Over de 11 G...,11 Degrees Azul Core Pull Over Hoodie\nA estre...,11 Degrees,Azul,es


,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train


### Interpretación de los datos
Una fila de `examples` significa “para esta consulta, este producto fue uno de los candidatos y recibió esta etiqueta”. `query_id` agrupa candidatos de la misma consulta. `product_id` conecta con la ficha en `products`. Etiquetas: E exacto, S sustituto, C complemento, I irrelevante. **No se incluirán etiquetas en el índice ni en el contexto del asistente.**

In [3]:
# Explorar un ejemplo completo y comprobar el join
locale_vals = products.product_locale.dropna().unique().tolist()
print('Locales:', locale_vals)
# Ajusta el locale tras inspeccionar el resultado; en ESCI suele ser 'es_ES'.
LOCALE = next((x for x in locale_vals if str(x).lower().startswith('es')), locale_vals[0])
products_es = products[products.product_locale == LOCALE].copy()
examples_es = examples[examples.product_locale == LOCALE].copy()
print('Locale elegido:', LOCALE, '| productos:', len(products_es), '| ejemplos:', len(examples_es))
qid = examples_es.query_id.iloc[0]
joined = examples_es[examples_es.query_id == qid].merge(products_es, on=['product_locale','product_id'], how='left')
display(joined[['query','product_id','esci_label','product_title','product_brand']].head(10))


Locales: ['es', 'us', 'jp']
Locale elegido: es | productos: 260011 | ejemplos: 356410


,query,product_id,esci_label,product_title,product_brand
0,!solid camiseta sin manga,B08TQH4RBB,E,Tuc Tuc Punto Tropicool Camiseta sin Mangas pa...,Tuc Tuc
1,!solid camiseta sin manga,B08H2H18DN,E,"Esprit Maternity T-Shirt Nursing SL Camiseta, ...",Esprit Maternity
2,!solid camiseta sin manga,B08HJQJT2T,E,Aloha playas de Hawai Hawaii gafas de sol dive...,Funny Beach Tees
3,!solid camiseta sin manga,B08LP68XTS,S,Aloha Hawaii Piña Con Gafas De Sol Regalo De P...,Glitch Beach - Aloha Hawaii Tees
4,!solid camiseta sin manga,B08NDS1HS2,E,Zereesa Camiseta sin Mangas de Punto con Cuell...,Zereesa
5,!solid camiseta sin manga,B08PNZHW6T,S,Sykooria 3 Piezas Camisetas Manga Larga Hombre...,Sykooria
6,!solid camiseta sin manga,B08TQF34WG,I,Tuc Tuc Punto Draw A Rex Camiseta sin Mangas p...,Tuc Tuc
7,!solid camiseta sin manga,B08TR31689,E,Tuc Tuc Punto KAMOGAWA Camiseta sin Mangas par...,Tuc Tuc
8,!solid camiseta sin manga,B0983KNTQX,E,"MEETWEE Camisetas Tirantes Hombre, Camisa sin ...",MEETWEE
9,!solid camiseta sin manga,B08ZC4CHDH,S,frueo 3 Piezas Camisetas Manga Corta Hombre Ca...,frueo


## 3. Preparar catálogo y datos de evaluación
Para mantener el notebook liviano, indexaremos hasta 5 000 productos en español. Separamos consultas por `query_id`; primero usamos train para desarrollo y test queda reservado para evaluación final.

In [4]:
MAX_PRODUCTS = 5000
products_es = products_es.drop_duplicates(['product_locale','product_id']).head(MAX_PRODUCTS).reset_index(drop=True)
TEXT_COLS = ['product_title','product_description','product_bullet_point','product_brand','product_color']
for c in TEXT_COLS:
    if c not in products_es: products_es[c] = ''
    products_es[c] = products_es[c].fillna('').astype(str)
products_es['document'] = products_es.apply(lambda r: '\n'.join(f'{c}: {r[c]}' for c in TEXT_COLS if r[c].strip()), axis=1)
# Evaluación solo para pares cuyo producto está dentro del catálogo indexado
examples_es = examples_es[examples_es.product_id.isin(products_es.product_id)].copy()
train = examples_es[examples_es.split == 'train'].copy()
test = examples_es[examples_es.split == 'test'].copy()
print('Productos indexables:', len(products_es), 'query_id train:', train.query_id.nunique(), 'test:', test.query_id.nunique())


Productos indexables: 5000 query_id train: 362 test: 133


## 4. RAG básico: embeddings e índice FAISS
Se usa un modelo multilingüe de embeddings. El mismo modelo transforma productos y consultas. FAISS almacena vectores normalizados y devuelve los productos más similares.

In [5]:
from sentence_transformers import SentenceTransformer
import faiss

EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
embedder = SentenceTransformer(EMBED_MODEL)
texts = products_es.document.tolist()
embeddings = embedder.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True).astype('float32')
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print('Vectores indexados:', index.ntotal)


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Vectores indexados: 5000


In [6]:
def retrieve(query, k=10):
    qv = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, ids = index.search(qv, min(k, len(products_es)))
    out = products_es.iloc[ids[0]].copy()
    out['score'] = scores[0]
    return out

display(retrieve('auriculares para escuchar música', 5)[['product_id','product_title','score']])


,product_id,product_title,score
1292,B075FMK7SD,"Aftershokz Trekz Air, Auriculares Bluetooth In...",0.671051
4752,B06XWXR6M9,Auriculares con Cable Auriculares In Ear Sonid...,0.668992
1718,B00DCXWXEI,AKG K712 Pro - Open Back - Auriculares para DJ...,0.621813
4820,B07RQVTPP7,"JVC HA-S35BT - Auriculares (Inalámbrico, Diade...",0.609823
2827,B00U1JQN5Q,PG Audio Kit de instalación de altavoces para ...,0.607711


## 5. Baseline léxico y evaluación nDCG@10
Usamos TF-IDF como baseline léxico. La métrica usa juicios ESCI solo en esta celda de evaluación, nunca en `retrieve()` ni en el prompt del asistente.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import normalize

vectorizer = TfidfVectorizer(max_features=100000, ngram_range=(1,2), min_df=2)
tfidf_products = vectorizer.fit_transform(products_es.document)
product_ids = products_es.product_id.tolist()
id_to_row = {pid:i for i,pid in enumerate(product_ids)}
def lexical_retrieve(query, k=10):
    qv = vectorizer.transform([query]); sims = linear_kernel(qv, tfidf_products).ravel()
    inds = np.argsort(sims)[::-1][:k]
    return [(product_ids[i], float(sims[i])) for i in inds]

REL = {'E':1.0, 'S':0.1, 'C':0.01, 'I':0.0}
def ndcg_at_k(ranked_ids, judged, k=10):
    gains = [REL.get(judged.get(pid,'I'),0.0) for pid in ranked_ids[:k]]
    dcg = sum((2**g-1)/np.log2(i+2) for i,g in enumerate(gains))
    ideal = sorted([REL.get(v,0.0) for v in judged.values()], reverse=True)[:k]
    idcg = sum((2**g-1)/np.log2(i+2) for i,g in enumerate(ideal))
    return dcg/idcg if idcg else 0.0

def eval_retriever(frame, mode='dense', limit=200):
    qids = frame["query_id"].drop_duplicates().head(limit)
    vals = []
    for qid in qids:
        g = frame[frame["query_id"] == qid]
        query_text = g["query"].iloc[0]
        judged = dict(zip(g["product_id"], g["esci_label"]))

        if mode == 'dense':
            ranked = retrieve(query_text, 10)["product_id"].tolist()
        else:
            ranked = [
                pid for pid, _ in lexical_retrieve(query_text, 10)
            ]
        vals.append(ndcg_at_k(ranked, judged, 10))
    return float(np.mean(vals)), len(vals)

# Desarrollo con train; evaluación final ejecutar con test una sola vez.
print('Train dense:', eval_retriever(train,'dense'))
print('Train lexical:', eval_retriever(train,'lexical'))


Train dense: (0.274134375887351, 200)
Train lexical: (0.48658172789377796, 200)


## 6. Tienda sintética reproducible y herramientas
Los precios y existencias no vienen de ESCI: los generamos con semilla fija. Las funciones validan cantidades, presupuesto y stock. Confirmar vuelve a validar y usa `confirmation_id` para no duplicar pedidos.

In [8]:
def restore_store(seed=SEED):
    rng=np.random.default_rng(seed)
    store=products_es[['product_id','product_title','product_brand']].copy()
    store['price']=np.round(rng.uniform(8,250, len(store)),2)
    store['stock']=rng.integers(0,21,len(store))
    return store.set_index('product_id').to_dict('index')

store=restore_store()
cart={}
orders={}
confirmations={}
def product_search(query,k=5): return retrieve(query,k)
def get_product(pid): return store.get(pid)
def set_quantity(pid, qty):
    if pid not in store: return {'ok':False,'message':'Producto no encontrado'}
    if not isinstance(qty,int) or qty<=0: return {'ok':False,'message':'Cantidad debe ser entero positivo'}
    if qty>store[pid]['stock']: return {'ok':False,'message':f"Stock insuficiente: quedan {store[pid]['stock']}"}
    cart[pid]=qty; return {'ok':True,'message':'Carrito actualizado'}
def quote(budget=None):
    lines=[{'product_id':p,'title':store[p]['product_title'],'qty':q,'price':store[p]['price'],'subtotal':round(q*store[p]['price'],2)} for p,q in cart.items()]
    total=round(sum(x['subtotal'] for x in lines),2)
    return {'lines':lines,'total':total,'within_budget': budget is None or total<=budget}
def confirm_order(confirmation_id, budget=None):
    if confirmation_id in confirmations: return confirmations[confirmation_id]
    q=quote(budget)
    if not q['lines']: return {'ok':False,'message':'Carrito vacío'}
    if not q['within_budget']: return {'ok':False,'message':'La cotización supera el presupuesto'}
    for pid,qty in cart.items():
        if store[pid]['stock']<qty: return {'ok':False,'message':f"Stock cambió para {pid}; no se registró el pedido"}
    # Actualización agrupada tras validar todas las líneas
    order_id=str(uuid.uuid4())
    for pid,qty in cart.items(): store[pid]['stock']-=qty
    result={'ok':True,'order_id':order_id,'quote':q}
    orders[order_id]=result; confirmations[confirmation_id]=result
    return result


## 7. Modelo local para redactar respuestas con contexto recuperado

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
LLM_ID='Qwen/Qwen2.5-1.5B-Instruct'
tokenizer=AutoTokenizer.from_pretrained(LLM_ID)
llm=AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype='auto', device_map='auto')
def answer_with_rag(question, docs):
    context='\n'.join(f"- {r.product_title}. {r.product_description or ''} Marca: {r.product_brand or 'desconocida'}. Color: {r.product_color or 'desconocido'}. ID: {r.product_id}" for r in docs.itertuples())
    messages=[{'role':'system','content':'Responde en español usando solo el contexto. Si falta un dato, dilo. No inventes precios ni stock.'},{'role':'user','content':f'Contexto:\n{context}\n\nPregunta: {question}'}]
    prompt=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs=tokenizer(prompt, return_tensors='pt').to(llm.device)
    output=llm.generate(**inputs, max_new_tokens=180, do_sample=False)
    return tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


## 8. Agente mínimo con herramientas explícitas y estado conversacional
El modelo redacta; el control de operaciones queda en código. Esta versión sencilla admite búsqueda, añadir producto por ID, cotizar y confirmar; si la petición no indica producto/cantidad, solicita aclaración. No delega decisiones de inventario al LLM.

In [10]:
conversation={'preferences':{},'budget':None,'last_results':None,'transcript':[]}
def assistant_turn(text, confirm=False):
    conversation['transcript'].append({'role':'user','text':text})
    low=text.lower()
    # Extraer presupuesto de forma básica
    m=re.search(r'(?:presupuesto|máximo|hasta)\s*(?:de)?\s*(\d+(?:[.,]\d+)?)',low)
    if m: conversation['budget']=float(m.group(1).replace(',','.'))
    if any(w in low for w in ['buscar','busco','recomienda','quiero','auricular','celular','zapatilla','producto']):
        res=product_search(text,5); conversation['last_results']=res
        response=answer_with_rag(text,res)
    elif 'cotiza' in low or 'cotización' in low:
        response=json.dumps(quote(conversation['budget']),ensure_ascii=False)
    elif confirm:
        response=json.dumps(confirm_order(str(uuid.uuid4()),conversation['budget']),ensure_ascii=False)
    else:
        response='¿Qué producto buscas y qué cantidad necesitas? Puedes indicar también un presupuesto.'
    conversation['transcript'].append({'role':'assistant','text':response})
    return response


## 9. Ablación sencilla
Ablación: comparar recuperación densa con recuperación léxica, sobre el mismo conjunto de consultas. Reportamos nDCG@10. No se ajustan parámetros usando test.

In [11]:
# Ejecutar al final y una sola vez para test
print('Test dense:', eval_retriever(test,'dense',limit=200))
print('Test lexical:', eval_retriever(test,'lexical',limit=200))


Test dense: (0.25034932588849546, 133)
Test lexical: (0.4865882630633041, 133)


## 10. ASR en español y prueba de corrección de transcripción

In [12]:
from faster_whisper import WhisperModel
asr=WhisperModel('small', device=DEVICE, compute_type='float16' if DEVICE == 'cuda' else 'int8')
def transcribe(path):
    segments,info=asr.transcribe(path, language='es', vad_filter=True)
    return ' '.join(s.text.strip() for s in segments)
# Usa una ruta local a un archivo WAV; revisa la transcripción antes de enviarla al asistente.


## 11. Cargar perfiles de voz disponibles

El notebook `02_preparar_perfil_de_voz.ipynb` crea un archivo `.pt` por persona en `data/voice_profiles/`. El asistente carga los perfiles disponibles y permite seleccionar uno al llamar a `speak()`.

In [13]:
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import XttsAudioConfig
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.api import TTS

# Configuración requerida por la carga del checkpoint oficial de XTTS-v2.
torch.serialization.add_safe_globals([XttsConfig, XttsAudioConfig, BaseDatasetConfig])
os.environ["COQUI_TOS_AGREED"] = "1"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

tts = TTS(
    "tts_models/multilingual/multi-dataset/xtts_v2",
    gpu=DEVICE == "cuda",
)
model = tts.synthesizer.tts_model
print("XTTS cargado")
VOICE_PROFILES_DIR = PROJECT_DIR / "data" / "voice_profiles"
assert VOICE_PROFILES_DIR.is_dir(), f"No existe la carpeta: {VOICE_PROFILES_DIR}"

# Carga todos los perfiles .pt de la carpeta; el nombre del archivo identifica a la persona.
voice_profiles = {
    path.stem: torch.load(path, map_location=DEVICE, weights_only=True)
    for path in VOICE_PROFILES_DIR.glob("*.pt")
}
assert voice_profiles, f"No hay perfiles .pt en {VOICE_PROFILES_DIR}"
print("Perfiles disponibles:", ", ".join(sorted(voice_profiles)))

VOICE_ID = "winz"  # Cambia este valor para elegir la voz predeterminada.
assert VOICE_ID in voice_profiles, f"Perfil no encontrado: {VOICE_ID}"

def speak(text, out=str(ARTIFACTS / "respuesta.wav"), voice_id=VOICE_ID):
    if voice_id not in voice_profiles:
        raise ValueError(f"Perfil no disponible: {voice_id}. Opciones: {list(voice_profiles)}")
    profile = voice_profiles[voice_id]
    with torch.inference_mode():
        result = model.inference(
            text=text, language="es",
            gpt_cond_latent=profile["gpt_cond_latent"].to(DEVICE),
            speaker_embedding=profile["speaker_embedding"].to(DEVICE),
            enable_text_splitting=True
        )
    import soundfile as sf
    sf.write(out, result["wav"], 24000)
    return out

# Ejemplo: speak("Hola, ¿en qué puedo ayudarte?", voice_id="winz")

/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/api.py:70: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")


 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/tts/layers/xtts/xtts_manager.py:5: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.speakers = torch.load(speaker_file_path)
/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/utils/io.py:54: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  return torch.load(f, map_location=map_location, **kwargs)


XTTS cargado
Perfiles disponibles: ana, diego, elsa, estuardo, nicole, winz


## 12. Escenarios reproducibles y registro
El laboratorio solicita al menos 20 escenarios operativos, seis con audio y varias intervenciones en al menos tres. La tabla permite registrar resultado esperado/obtenido y evidencias. Ejecutar pruebas después de restaurar tienda y conversación en cada caso.

In [16]:
SCENARIOS=[
{'id':'S01','type':'texto','case':'Compra dentro de restricciones','expected':'Pedido confirmado'},
{'id':'S02','type':'texto','case':'Consulta ambigua','expected':'Solicita aclaración'},
{'id':'S03','type':'texto','case':'Cambio de cantidad','expected':'Conserva resto de preferencias'},
{'id':'S04','type':'texto','case':'Producto agotado','expected':'Ofrece alternativa o informa'},
{'id':'S05','type':'audio','case':'Transcripción corregible','expected':'Pide confirmar texto antes de comprar'},
{'id':'S06','type':'audio','case':'Stock cambia y confirmación repetida','expected':'No pedido parcial ni duplicado'},
]
# Completar hasta 20 escenarios con variantes; registrar las conversaciones completas en JSONL.
def reset_session():
    global store,cart,orders,confirmations,conversation
    store=restore_store(SEED); cart={}; orders={}; confirmations={}
    conversation={'preferences':{},'budget':None,'last_results':None,'transcript':[]}

def save_record(record, path=ARTIFACTS/'scenario_log.jsonl'):
    with open(path,'a',encoding='utf-8') as f: f.write(json.dumps(record,ensure_ascii=False)+'\n')


## 13. Métricas y resultados a reportar
Completar la matriz de 20 escenarios; analizar cinco aciertos y cinco fallos. Para voz guardar audio, transcripción original/corregida, llamadas a herramientas, respuesta y estado final. Medir tiempos de carga inicial y de consulta por separado.

In [ ]:
results={
'retrieval_ndcg10_dense': None,
'retrieval_ndcg10_lexical': None,
'scenario_success_rate': None,
'restriction_violations': None,
'asr_errors_quantities_amounts_products': None,
'latency_by_stage_seconds': {},
'voice_intelligibility_notes': None,
'voice_similarity_notes': None,
}
print(json.dumps(results,indent=2,ensure_ascii=False))


{
  "retrieval_ndcg10_dense": null,
  "retrieval_ndcg10_lexical": null,
  "scenario_success_rate": null,
  "restriction_violations": null,
  "asr_errors_quantities_amounts_products": null,
  "latency_by_stage_seconds": {},
  "voice_intelligibility_notes": null,
  "voice_similarity_notes": null
}


## Límites conocidos del MVP
- Se indexa un máximo de 5 000 productos, seleccionados por orden del archivo; documentar este criterio o muestrear con semilla.
- La extracción de intención/presupuesto es básica; ampliar únicamente con casos de prueba que lo requieran.
- Para robustez de producción, las transacciones de tienda deben estar en BD transaccional; aquí se simulan en memoria con restauración determinista.
- XTTS puede requerir ajuste de versión y una referencia de voz válida. Si falla, documentar la limitación; no sustituir por una API externa.
